In [1]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


https://repo1.maven.org/maven2/ added as a remote repository with the name: repo-1
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b47abbf5-3ae9-44d3-9e3d-c38c4b0464c5;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorprone#error_prone_annotations;2.2.0 in central
	found com.google.j2objc#j2objc-annotations;1.1 in central
	found org.codehaus.mojo#animal-sniffer-annotations;1.17 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-guava;1.1.1 in central
	found org.eclipse.jetty#jetty-util-ajax;9.4.43.v20210629 in central
	found org.

Spark session initialized successfully....✅ 
spark v3.5.7 connected .... ✅
ADLS Gen2 storage account 'clientdatastorage' Connected ....✅ 
Created 4 tables in 'bupa_bronze' database
Created 4 tables in 'bupa_bronze' database
Created 4 tables in 'bupa_silver' database
Created 4 tables in 'bupa_silver' database
Created 33 tables in 'bupa_gold' database
Created 33 tables in 'bupa_gold' database


In [2]:
import sys
import importlib
import os

# Add the Silver layer directory to sys.path BEFORE importing
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
import utils_silver
from utils_silver import *

importlib.reload(utils_silver)

# Import config and utilities
PROJECT_ROOT = "/Users/manojrammopati/Public/Projects/bupa_insurance_project"
sys.path.insert(0, os.path.join(PROJECT_ROOT, "config"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

import config
import ml_utils
import data_utils

importlib.reload(config)
importlib.reload(ml_utils)
importlib.reload(data_utils)

path_bronze, path_silver, path_gold = utils_silver.build_paths(storage_account1)
print("Imported utils_silver from", utils_silver.__file__)
print("Imported config from", config.__file__)
print("Imported ml_utils from", ml_utils.__file__)
print("Imported data_utils from", data_utils.__file__)

✅ utils_silver.py loaded
✅ utils_silver.py loaded
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py
Imported config from /Users/manojrammopati/Public/Projects/bupa_insurance_project/config/config.py
Imported ml_utils from /Users/manojrammopati/Public/Projects/bupa_insurance_project/src/ml_utils.py
Imported data_utils from /Users/manojrammopati/Public/Projects/bupa_insurance_project/src/data_utils.py
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py
Imported config from /Users/manojrammopati/Public/Projects/bupa_insurance_project/config/config.py
Imported ml_utils from /Users/manojrammopati/Public/Projects/bupa_insurance_project/src/ml_utils.py
Imported data_utils from /Users/manojrammopati/Public/Projects/bupa_insurance_project/src/data_utils.py


#  Intro
 ML – Claims High-Cost Risk Model

 **Goal:** Train and evaluate Spark ML models to predict *high-cost claims* using
 the curated Gold feature table `ft_claims_risk_split`.

 Label: `Is_High_Cost` (0 = normal, 1 = high cost)

 Steps:
 1. Load Gold feature table
 2. Clean + train/test split
 3. Build Spark ML pipeline (index + encode + assemble)
 4. Train several models (LR, RF, GBT)
 5. Persist the best model
 6. Score sample claims with high-cost probability


# Cell 1 — MLflow setup for high-cost model

In [3]:
# Cell 0b — MLflow setup
from pathlib import Path
import mlflow
import mlflow.spark

PROJECT_ROOT = Path("/Users/manojrammopati/Public/Projects/bupa_insurance_project")

# File-based tracking store inside the repo
TRACKING_URI = f"file:{PROJECT_ROOT / 'mlruns'}"
mlflow.set_tracking_uri(TRACKING_URI)

# One experiment per use case
USE_CASE = "claims_risk_high_cost"  # High-cost part of claims_risk
mlflow.set_experiment(f"bupa_{USE_CASE}")

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", f"bupa_{USE_CASE}")


MLflow tracking URI: file:/Users/manojrammopati/Public/Projects/bupa_insurance_project/mlruns
Experiment: bupa_claims_risk_high_cost


/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:177: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance.
  return FileStore(store_uri, store_uri)


# Cell 2 – Imports & config

In [4]:
# Cell 2 — Load ft_claims_risk_split from golddata using config

STORAGE_ACCOUNT = storage_account1
CONTAINER_GOLD = "golddata"
DB_GOLD = "bupa_gold"

# Construct the path to ft_claims_risk_split
FT_CLAIMS_RISK_SPLIT_PATH = (
    f"abfss://{CONTAINER_GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    "ft_claims_risk_split"
)

print("Reading from:", FT_CLAIMS_RISK_SPLIT_PATH)

claims_all = (
    spark.read
         .format("delta")
         .load(FT_CLAIMS_RISK_SPLIT_PATH)
)

print("Total rows:", claims_all.count())

claims_all.groupBy("dataset_split").count().show()

claims_all.printSchema()

claims_all.show(5, truncate=False)

Reading from: abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_claims_risk_split


Total rows: 558211
+-------------+------+
|dataset_split| count|
+-------------+------+
|        train|446102|
|         test|112109|
+-------------+------+

root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Type_Code: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Payout_to_Amount_Ratio: double (nullable = true)
 |-- Days_To_Settle: integer (nullable = true)
 |-- High_Cost_Claim_Flag: integer (nullable = true)
 |-- Claim_Fraud_Label: integer (nullable = true)
 |-- Provider_Fraud_Label: integer (nullable = true)
 |-- Provider_Risk_Tier: string (nullable = true)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_date_valid: integer (nullable = true)
 |-- Is_Fraudulent_Claim: integer (nullable = true)
 |-- Is_High_Cost: integer (null

+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Type_Code|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Days_To_Settle|High_Cost_Claim_Flag|Claim_Fraud_Label|Provider_Fraud_Label|Provider_Risk_Tier|dq_money_valid|dq_date_valid|Is_Fraudulent_Claim|Is_High_Cost|dataset_split|
+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |TRAVEL         |Settled     |244.07511199

# Cell 3 – Load feature table

In [5]:
# Cell 3 — Separate train and test splits using stratified sampling (Priority 2)

# Priority 2: Use stratified split if enabled in config
use_stratified = config.ML_CONFIG.get("use_stratified_split", False)

if use_stratified:
    # Priority 2: Stratified split maintains label distribution
    train_df = claims_all.filter(F.col("dataset_split") == "train").cache()
    test_df = claims_all.filter(F.col("dataset_split") == "test").cache()
    print("✅ Using pre-split data (stratified by dataset_split column)")
else:
    # Fallback to simple split
    train_df = claims_all.filter(F.col("dataset_split") == "train").cache()
    test_df = claims_all.filter(F.col("dataset_split") == "test").cache()
    print("⚠️ Using pre-split data (not stratified)")

print("Train rows:", train_df.count())
print("Test rows :", test_df.count())

# Quick sanity check on label distribution
for name, df in [("train", train_df), ("test", test_df)]:
    print(f"\n{name} label distribution:")
    (
        df.groupBy("Is_High_Cost")
          .count()
          .orderBy("Is_High_Cost")
          .show()
    )


✅ Using pre-split data (stratified by dataset_split column)


Train rows: 446102


Test rows : 112109

train label distribution:
+------------+------+
|Is_High_Cost| count|
+------------+------+
|           0|400391|
|           1| 45711|
+------------+------+


test label distribution:
+------------+------+
|Is_High_Cost| count|
+------------+------+
|           0|100725|
|           1| 11384|
+------------+------+

+------------+------+
|Is_High_Cost| count|
+------------+------+
|           0|400391|
|           1| 45711|
+------------+------+


test label distribution:
+------------+------+
|Is_High_Cost| count|
+------------+------+
|           0|100725|
|           1| 11384|
+------------+------+



# Cell 4 – Filter label + train/test

In [6]:
# Cell 4 — Define feature columns & null-handling using config

# First, import required PySpark ML classes
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

# Use feature lists from config (Priority 5: Centralized Config)
FEATURE_CONFIG = config.FEATURE_ENGINEERING["claims_risk"]
numeric_cols = FEATURE_CONFIG["numeric_features"]
categorical_cols = FEATURE_CONFIG["categorical_features"]
label_col = "Is_High_Cost"  # High-cost-specific label
null_handling = FEATURE_CONFIG.get("null_handling", {"numeric": 0.0, "categorical": "Unknown"})

print(f"🔹 Using features from config.FEATURE_ENGINEERING['claims_risk']")
print("Numeric features :", numeric_cols)
print("Categorical feats:", categorical_cols)
print("Label column     :", label_col)

# 2) Basic null handling using config strategy
def prep_nulls(df):
    df_num_filled = df
    numeric_fill = null_handling.get("numeric", 0.0)
    categorical_fill = null_handling.get("categorical", "Unknown")
    
    for c in numeric_cols:
        if c in df_num_filled.columns:
            df_num_filled = df_num_filled.withColumn(
                c,
                F.coalesce(F.col(c), F.lit(numeric_fill))
            )
    df_cat_filled = df_num_filled
    for c in categorical_cols:
        if c in df_cat_filled.columns:
            df_cat_filled = df_cat_filled.withColumn(
                c,
                F.coalesce(F.col(c).cast("string"), F.lit(categorical_fill))
            )
    # ensure label is numeric 0/1
    df_final = df_cat_filled.withColumn(label_col, F.col(label_col).cast("double"))
    return df_final

# Apply null handling AND drop rows with missing/invalid labels
train_pre = (
    prep_nulls(train_df)
    .filter(F.col(label_col).isNotNull())
    .filter(F.col(label_col).isin(0.0, 1.0))
)

test_pre = (
    prep_nulls(test_df)
    .filter(F.col(label_col).isNotNull())
    .filter(F.col(label_col).isin(0.0, 1.0))
)

print("After cleaning:")
print("  train_pre rows:", train_pre.count())
print("  test_pre rows :", test_pre.count())

# 3) Build indexers + one-hot encoders
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_oh"
    )
    for c in categorical_cols
]

# 4) VectorAssembler for all features
feature_cols = numeric_cols + [f"{c}_oh" for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

print("Total features col list length:", len(feature_cols))

🔹 Using features from config.FEATURE_ENGINEERING['claims_risk']
Numeric features : ['Claim_Amount_GBP', 'Payout_GBP', 'Payout_to_Amount_Ratio', 'Days_To_Settle']
Categorical feats: ['Claim_Type_Name', 'Claim_Status', 'Claim_Type_Code', 'Provider_Risk_Tier', 'Provider_ID']
Label column     : Is_High_Cost
After cleaning:
After cleaning:
  train_pre rows: 446102
  test_pre rows : 112109
Total features col list length: 9
  train_pre rows: 446102
  test_pre rows : 112109
Total features col list length: 9


# Cell 5 – Feature columns & null handling

In [7]:
# Cell 5 — Define three ML pipelines with hyperparameters from config (Priority 4)

# Get hyperparameters from config
ml_config = config.ML_CONFIG["algorithms"]

# Priority 2: Calculate class weights if enabled
use_class_weights = config.ML_CONFIG.get("use_class_weights", False)
class_weights = {}

if use_class_weights:
    print("🔹 Computing class weights (Priority 2)...")
    class_weights = ml_utils.MLPipeline(spark, "bupa_claims_risk_high_cost", USE_CASE, config.__dict__).compute_class_weights(
        train_pre, label_col
    )
    print(f"  Class weights: {class_weights}")

# Logistic Regression with config parameters
lr_params = ml_config.get("LogisticRegression", {})
lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=lr_params.get("maxIter", 100),
    regParam=lr_params.get("regParam", 0.01),
    elasticNetParam=lr_params.get("elasticNetParam", 0.0),
)

pipeline_lr = Pipeline(stages=indexers + encoders + [assembler, lr])

# Random Forest with config parameters and optional class weights
rf_params = ml_config.get("RandomForestClassifier", {})
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_col,
    numTrees=rf_params.get("numTrees", 100),
    maxDepth=rf_params.get("maxDepth", 8),
    subsamplingRate=rf_params.get("subsamplingRate", 0.8),
    featureSubsetStrategy=rf_params.get("featureSubsetStrategy", "auto"),
    seed=config.ML_CONFIG.get("random_seed", 42),
)

pipeline_rf = Pipeline(stages=indexers + encoders + [assembler, rf])

# Gradient Boosted Trees with config parameters
gbt_params = ml_config.get("GBTClassifier", {})
gbt = GBTClassifier(
    featuresCol="features",
    labelCol=label_col,
    maxIter=gbt_params.get("maxIter", 80),
    maxDepth=gbt_params.get("maxDepth", 5),
    stepSize=gbt_params.get("stepSize", 0.05),
    seed=config.ML_CONFIG.get("random_seed", 42)
)

pipeline_gbt = Pipeline(stages=indexers + encoders + [assembler, gbt])

print("✅ Pipelines defined with config parameters:")
print("   - pipeline_lr (Logistic Regression)")
print("   - pipeline_rf (Random Forest)")
print("   - pipeline_gbt (Gradient Boosted Trees)")


🔹 Computing class weights (Priority 2)...
  Class weights: {0: 0.5570829514149919, 1: 4.879591345627967}
✅ Pipelines defined with config parameters:
   - pipeline_lr (Logistic Regression)
   - pipeline_rf (Random Forest)
   - pipeline_gbt (Gradient Boosted Trees)
  Class weights: {0: 0.5570829514149919, 1: 4.879591345627967}
✅ Pipelines defined with config parameters:
   - pipeline_lr (Logistic Regression)
   - pipeline_rf (Random Forest)
   - pipeline_gbt (Gradient Boosted Trees)


# Cell 6 – Build pipeline skeleton

In [8]:
# Cell 6 — (Skipped - already done in Cell 4)
# The indexers, encoders, and assembler are already built in Cell 4
# This avoids duplication and variable naming issues

print("✅ Indexers, encoders, and assembler already configured in Cell 4")
print("Ready to train models!")

✅ Indexers, encoders, and assembler already configured in Cell 4
Ready to train models!


# Cell 7 – Train & eval helper

In [9]:
# Cell 7 — Evaluation helper function

def evaluate_binary(pred_df, label_col=label_col, prediction_col="prediction", prob_col="probability"):
    # AUC ROC
    evaluator_roc = BinaryClassificationEvaluator(
        labelCol=label_col,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    auc_roc = evaluator_roc.evaluate(pred_df)

    # AUC PR
    evaluator_pr = BinaryClassificationEvaluator(
        labelCol=label_col,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR"
    )
    auc_pr = evaluator_pr.evaluate(pred_df)

    # F1 & Accuracy (treat as multiclass)
    f1_eval = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol=prediction_col,
        metricName="f1"
    )
    f1 = f1_eval.evaluate(pred_df)

    acc_eval = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol=prediction_col,
        metricName="accuracy"
    )
    acc = acc_eval.evaluate(pred_df)

    return {
        "auc_roc": auc_roc,
        "auc_pr": auc_pr,
        "f1": f1,
        "accuracy": acc
    }

print("✅ evaluate_binary function defined")

✅ evaluate_binary function defined


# Cell 8 – Define models & train

In [10]:
# Cell 8 — Train LR and RF only (skipping GBT)

results = []

# Get MLflow run tracking
store_feature_importance = config.ML_CONFIG.get("model_versioning", {}).get("store_feature_importance", False)

for pipeline, model_name in [
    (pipeline_lr, "LogisticRegression"),
    (pipeline_rf, "RandomForest"),
]:
    with mlflow.start_run(run_name=f"{USE_CASE}_{model_name}"):
        print(f"\n===== Training {model_name} =====")
        model = pipeline.fit(train_pre)
        
        print(f"Scoring test set with {model_name} ...")
        pred = model.transform(test_pre)
        
        metrics = evaluate_binary(pred)
        
        print(f"{model_name} metrics:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")
        
        # Priority 1: Log to MLflow
        mlflow.log_params({
            "model_type": model_name,
            "use_case": USE_CASE,
            "label_column": label_col,
            "use_class_weights": use_class_weights,
            **({f"class_weight_{int(k)}": round(v, 4) for k, v in class_weights.items()} if class_weights else {})
        })
        
        mlflow.log_metrics(metrics)
        
        # Priority 8: Extract and log feature importance
        if store_feature_importance and model_name in ["RandomForest"]:
            try:
                # Get the trained classifier from the pipeline
                trained_clf = model.stages[-1]
                
                # Get feature names (numeric + encoded categorical)
                all_feature_names = numeric_cols + [f"{c}_oh" for c in categorical_cols]
                
                # Extract feature importance
                feature_importance = ml_utils.MLPipeline(
                    spark, f"bupa_{USE_CASE}", USE_CASE, config.__dict__
                ).get_feature_importance(
                    trained_clf,
                    all_feature_names,
                    top_n=10
                )
                
                # Log top features
                for feature, importance in feature_importance.items():
                    mlflow.log_metric(f"feature_importance_{feature}", importance)
                
                print(f"✅ Logged top-10 feature importance for {model_name}")
            except Exception as e:
                print(f"⚠️ Could not extract feature importance: {e}")
        
        results.append((model_name, metrics, model, pred))

# Show as a small Spark table for convenience
metrics_rows = [
    (name, m["auc_roc"], m["auc_pr"], m["f1"], m["accuracy"])
    for name, m, _, _ in results
]

metrics_df = spark.createDataFrame(
    metrics_rows,
    schema="model STRING, auc_roc DOUBLE, auc_pr DOUBLE, f1 DOUBLE, accuracy DOUBLE"
)

print("\n🏆 Model comparison (LR vs RF only):")
metrics_df.show(truncate=False)


===== Training LogisticRegression =====


Scoring test set with LogisticRegression ...


LogisticRegression metrics:
  auc_roc: 0.9988
  auc_pr: 0.9908
  f1: 0.9609
  accuracy: 0.9644

===== Training RandomForest =====


Scoring test set with RandomForest ...


RandomForest metrics:
  auc_roc: 0.9774
  auc_pr: 0.9653
  f1: 0.8504
  accuracy: 0.8985
✅ Logged top-10 feature importance for RandomForest

🏆 Model comparison (LR vs RF only):


+------------------+------------------+------------------+------------------+------------------+
|model             |auc_roc           |auc_pr            |f1                |accuracy          |
+------------------+------------------+------------------+------------------+------------------+
|LogisticRegression|0.99884500233462  |0.9908373075249012|0.9609475241298535|0.9643650375973383|
|RandomForest      |0.9773834534480952|0.9653464962356131|0.8503996285567071|0.8984559669607257|
+------------------+------------------+------------------+------------------+------------------+



# Cell 9 – Choose best model & save

In [16]:
# Cell 9 — Pick best model (by ROC AUC) and save it with sequential versioning

# End any active MLflow runs
mlflow.end_run()

# Pick the model with the highest ROC AUC
best_name, best_metrics, best_model, best_pred = max(results, key=lambda t: t[1]["auc_roc"])
print(f"🏆 Best model by AUC ROC: {best_name}")
print(f"   Metrics: {best_metrics}")

# Priority 1: Use sequential model versioning (v1.0, v2.0, v3.0, etc.)
STORAGE_ACCOUNT = storage_account1
CONTAINER_GOLD = "golddata"
MODELS_BASE_PATH = (
    f"abfss://{CONTAINER_GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    f"models/{USE_CASE}"
)

# Auto-detect next version using Spark Hadoop FileSystem API
try:
    # Get Hadoop FileSystem
    hadoop_fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        spark._jvm.org.apache.hadoop.fs.Path(MODELS_BASE_PATH).toUri(),
        spark._jsc.hadoopConfiguration()
    )
    
    # List existing version folders
    path_obj = spark._jvm.org.apache.hadoop.fs.Path(MODELS_BASE_PATH)
    status_array = hadoop_fs.listStatus(path_obj)
    
    existing_versions = []
    for status in status_array:
        folder_name = status.getPath().getName()
        if folder_name.startswith('v') and folder_name[1:].replace('.', '').isdigit():
            try:
                version_num = float(folder_name[1:])
                existing_versions.append(version_num)
            except:
                pass
    
    next_version = max(existing_versions) + 1.0 if existing_versions else 1.0
    version_str = f"v{next_version:.1f}"
    print(f"✅ Auto-detected existing versions: {sorted(existing_versions)}")
    print(f"   Next version: {version_str}")
except Exception as e:
    print(f"⚠️ Could not detect existing versions: {e}")
    version_str = "v1.0"

MODEL_PATH = f"{MODELS_BASE_PATH}/{version_str}"

print(f"\nSaving best model with version: {version_str}")
print(f"Path: {MODEL_PATH}")

best_model.write().overwrite().save(MODEL_PATH)

print("✅ Model saved with version tracking")

# Log final model metadata to MLflow (log metrics instead, params are immutable)
mlflow.log_metrics({
    "final_auc_roc": round(best_metrics["auc_roc"], 4),
    "final_auc_pr": round(best_metrics["auc_pr"], 4),
    "final_f1": round(best_metrics["f1"], 4),
    "final_accuracy": round(best_metrics["accuracy"], 4),
})

print(f"✅ Final model metrics logged to MLflow")

🏆 Best model by AUC ROC: LogisticRegression
   Metrics: {'auc_roc': 0.99884500233462, 'auc_pr': 0.9908373075249012, 'f1': 0.9609475241298535, 'accuracy': 0.9643650375973383}
✅ Auto-detected existing versions: [1.0]
   Next version: v2.0

Saving best model with version: v2.0
Path: abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_risk_high_cost/v2.0
✅ Auto-detected existing versions: [1.0]
   Next version: v2.0

Saving best model with version: v2.0
Path: abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_risk_high_cost/v2.0


✅ Model saved with version tracking
✅ Final model metrics logged to MLflow


# Cell 10 – Load & score sample claims

In [12]:
# Cell 10 — Score sample claims with best model

print("🏆 Best Model:", best_name)
print("📊 Best Model Metrics:")
for metric, value in best_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Show sample predictions on test set
print("\n📝 Sample predictions (first 20 test claims):")
sample_predictions = best_pred.select(
    label_col,
    "probability",
    "prediction"
).limit(20).show(truncate=False)

print("\n✅ Model training and evaluation complete!")

🏆 Best Model: LogisticRegression
📊 Best Model Metrics:
  auc_roc: 0.9988
  auc_pr: 0.9908
  f1: 0.9609
  accuracy: 0.9644

📝 Sample predictions (first 20 test claims):


+------------+-----------------------------------------+----------+
|Is_High_Cost|probability                              |prediction|
+------------+-----------------------------------------+----------+
|0.0         |[0.8595499721929729,0.1404500278070271]  |0.0       |
|0.0         |[0.9648548542074058,0.0351451457925942]  |0.0       |
|0.0         |[0.9675168794246224,0.03248312057537761] |0.0       |
|0.0         |[0.8224920003237164,0.1775079996762836]  |0.0       |
|0.0         |[0.9602909434353598,0.03970905656464019] |0.0       |
|0.0         |[0.9833972009320484,0.01660279906795159] |0.0       |
|0.0         |[0.8777904275672954,0.12220957243270458] |0.0       |
|0.0         |[0.9018141238709616,0.0981858761290384]  |0.0       |
|1.0         |[0.5110401253432804,0.48895987465671964] |0.0       |
|0.0         |[0.9692294129598108,0.0307705870401892]  |0.0       |
|0.0         |[0.9545111224025795,0.045488877597420485]|0.0       |
|0.0         |[0.975937651948228,0.0240623480517

#  Cell 11 — MLflow logging for high-cost model

In [ ]:
# Cell 11 — Log full MLflow run with params, metrics, and model artifact

import mlflow
import mlflow.spark
from datetime import datetime

# Ensure any active runs are closed before starting a new one
mlflow.end_run()

with mlflow.start_run(run_name="high_cost_claims_training"):

    # Log dataset info
    mlflow.log_param("train_rows", train_df.count())
    mlflow.log_param("test_rows", test_df.count())

    # Log features & label
    mlflow.log_param("label_col", label_col)
    mlflow.log_param("numeric_features", ",".join(numeric_cols))
    mlflow.log_param("categorical_features", ",".join(categorical_cols))

    # Log metrics for each model (results is list of tuples: (name, metrics_dict, model, pred))
    for model_name, metrics, _, _ in results:
        prefix = model_name.replace(" ", "_")
        
        mlflow.log_metric(f"{prefix}_auc_roc", metrics["auc_roc"])
        mlflow.log_metric(f"{prefix}_auc_pr", metrics["auc_pr"])
        mlflow.log_metric(f"{prefix}_f1", metrics["f1"])
        mlflow.log_metric(f"{prefix}_accuracy", metrics["accuracy"])

    # Log winning model metadata
    mlflow.log_param("best_model", best_name)

    # Log model hyperparameters
    if best_name == "RandomForest":
        mlflow.log_param("numTrees", rf.getNumTrees())
        mlflow.log_param("maxDepth", rf.getOrDefault("maxDepth"))
        mlflow.log_param("algo", "RandomForest")

    elif best_name == "LogisticRegression":
        mlflow.log_param("maxIter", lr.getOrDefault("maxIter"))
        mlflow.log_param("regParam", lr.getOrDefault("regParam"))
        mlflow.log_param("algo", "LogisticRegression")

    # Log the actual Spark model
    mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path="high_cost_best_model",
        registered_model_name=None
    )

    print("✅ Logged MLflow run to experiment 'bupa_claims_risk_high_cost'")

print("👉 You can open MLflow UI with: mlflow ui --backend-store-uri", TRACKING_URI)

✅ Logged MLflow run to experiment 'bupa_claims_risk_high_cost'
👉 You can open MLflow UI with: mlflow ui --backend-store-uri file:/Users/manojrammopati/Public/Projects/bupa_insurance_project/mlruns


25/12/25 07:33:54 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE

# 📈 Cell 13 SAVING ALL MODELS METRICS TO ML_MONITORING

In [15]:
# Cell 13 — Save all model metrics to ML_monitoring table

from datetime import datetime
import utils_silver

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(storage_account1)

now_ts = datetime.utcnow()
rows = []

# results is list of tuples: (name, metrics_dict, model, pred)
for model_name, metrics, _, _ in results:
    rows.append({
        "model_name": model_name,
        "use_case": "claims_risk_high_cost",
        "dataset_name": "ft_claims_risk",
        "dataset_split": "test",
        "auc": float(metrics.get("auc_roc", 0.0)),
        "accuracy": float(metrics.get("accuracy", 0.0)),
        "f1": float(metrics.get("f1", 0.0)),
        "auc_pr": float(metrics.get("auc_pr", 0.0)),
        "ts": now_ts,
        "notes": "high-cost model sweep - LR vs RF comparison",
    })

utils_silver.write_ml_monitoring(
    spark=spark,
    rows=rows,
    paths_gold=paths_gold,
)

print(f"✅ Logged {len(rows)} model metrics to ml_monitoring")

[ML_MON] Wrote 2 rows to abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring
✅ Logged 2 model metrics to ml_monitoring
